# Lab 3 : Implementation of Greedy Algorithms

---


## Objective of the Lab

The objective of this lab is to understand, design, and implement various **Greedy** algorithms using Python. Through this lab, we aim to:

- Understand the working principle of the greedy strategy of algorithm design (making the locally optimal choice at each step).
- Implement classical greedy algorithms for optimization (Fractional Knapsack, Job Sequencing with Deadlines).
- Implement greedy algorithms for graph problems (Prim's, Kruskal's, and Dijkstra's algorithms).
- Analyze the time and space complexity of each implemented algorithm.
- Understand when a greedy choice leads to a globally optimal solution, and the conditions (greedy-choice property and optimal substructure) required for correctness.


## Title of the Study

**Implementation and Analysis of Greedy Algorithms:**
Fractional Knapsack Problem, Job Sequencing with Deadlines, Prim's Algorithm, Kruskal's Algorithm, and Dijkstra's Algorithm.


## Related Theory (General)

A **Greedy Algorithm** builds up a solution piece by piece, always choosing the option that looks best *at the current moment* (the locally optimal choice), without reconsidering previous choices. A greedy strategy produces a globally optimal solution only when the problem exhibits:

1. **Greedy-choice property** — a globally optimal solution can be reached by making a locally optimal (greedy) choice.
2. **Optimal substructure** — an optimal solution to the problem contains optimal solutions to its subproblems.

In this lab, we study 5 classical greedy algorithms:

1. Fractional Knapsack Problem
2. Job Sequencing with Deadlines
3. Prim's Algorithm (Minimum Spanning Tree)
4. Kruskal's Algorithm (Minimum Spanning Tree)
5. Dijkstra's Algorithm (Single Source Shortest Path)

Each algorithm is discussed below with its theory, diagram, source code, output, and complexity analysis.


---
## Q1. Program to Implement the Fractional Knapsack Problem

### Related Theory
In the **Fractional Knapsack Problem**, we are given `n` items, each with a weight $w_i$ and a value $v_i$, and a knapsack of capacity $W$. Unlike the 0/1 knapsack, items can be broken into fractions — we may take any fraction $0 \le x_i \le 1$ of an item.

The greedy strategy is to compute the **value-to-weight ratio** $v_i / w_i$ for every item, sort items in decreasing order of this ratio, and greedily pick items (or fractions of items) with the highest ratio first until the knapsack is full. Since items are divisible, this greedy choice is always optimal.

### Related Diagram (Flow)
```
        Start
          |
   Read items (weight, value), capacity W
          |
   Compute ratio = value / weight for each item
          |
   Sort items in decreasing order of ratio
          |
   +--------------------------------------------+
   | for each item (in sorted order):           | <---+
   |   if item.weight <= remaining capacity:    |     |
   |       take item fully                      |     |
   |   else:                                    |     |
   |       take fraction = remaining/weight     |     |
   |       break                                | ----+
   +--------------------------------------------+
          |
   Output total value obtained
          |
         End
```


In [1]:
def fractional_knapsack(items, capacity):
    """
    items: list of tuples (name, weight, value)
    capacity: knapsack capacity
    Returns: (max_value, selection) where selection is a list of
             (name, fraction_taken, value_taken)
    """
    # Sort items by value/weight ratio in decreasing order (greedy choice)
    items = sorted(items, key=lambda it: it[2] / it[1], reverse=True)

    remaining = capacity
    total_value = 0.0
    selection = []

    for name, weight, value in items:
        if remaining <= 0:
            break
        if weight <= remaining:
            # Take the whole item
            selection.append((name, 1.0, value))
            total_value += value
            remaining -= weight
        else:
            # Take only the fraction that fits
            fraction = remaining / weight
            taken_value = fraction * value
            selection.append((name, fraction, taken_value))
            total_value += taken_value
            remaining = 0

    return total_value, selection


# Driver / Test
if __name__ == "__main__":
    items = [("Item1", 10, 60), ("Item2", 20, 100), ("Item3", 30, 120)]
    capacity = 50

    max_value, selection = fractional_knapsack(items, capacity)

    print(f"Knapsack Capacity: {capacity}")
    print("Selected items (name, fraction taken, value contributed):")
    for name, frac, val in selection:
        print(f"  {name}: fraction={frac:.3f}, value={val:.2f}")
    print(f"Maximum value obtainable: {max_value:.2f}")


Knapsack Capacity: 50
Selected items (name, fraction taken, value contributed):
  Item1: fraction=1.000, value=60.00
  Item2: fraction=1.000, value=100.00
  Item3: fraction=0.667, value=80.00
Maximum value obtainable: 240.00


### Analysis of the Algorithm
- **Time Complexity:** $O(n \log n)$ — dominated by sorting the items by value/weight ratio; the greedy selection pass itself takes $O(n)$.
- **Space Complexity:** $O(n)$ — for storing the sorted list of items and the selection.
- **Remark:** Because items are divisible, the greedy approach is always **optimal** for the Fractional Knapsack Problem (unlike the 0/1 Knapsack Problem, where greedy fails and Dynamic Programming is required).


---
## Q2. Program to Implement Job Sequencing with Deadlines

### Related Theory
Given a set of `n` jobs, where each job has a **deadline** and a **profit**, and each job takes unit time to complete, the goal is to schedule jobs so that the **total profit is maximized**, subject to the constraint that only one job can be scheduled at a time and every scheduled job must finish before or on its deadline.

The greedy strategy is: sort jobs in **decreasing order of profit**, and for each job (greedily, most profitable first) try to schedule it in the **latest available free time slot** that is still before or on its deadline. This maximizes the number of profitable jobs that can be fit in.

### Related Diagram (Flow)
```
        Start
          |
   Read jobs (id, deadline, profit)
          |
   Sort jobs in decreasing order of profit
          |
   Create time slots [0 .. max_deadline-1], all free
          |
   +---------------------------------------------------+
   | for each job (in sorted order):                   | <---+
   |   for slot = min(deadline, maxSlots)-1 down to 0: |     |
   |       if slot is free:                            |     |
   |           assign job to slot; mark slot filled    |     |
   |           break                                   | ----+
   +---------------------------------------------------+
          |
   Output scheduled jobs and total profit
          |
         End
```


In [2]:
def job_sequencing(jobs):
    """
    jobs: list of tuples (job_id, deadline, profit)
    Returns: (scheduled_jobs, total_profit)
    """
    # Greedy choice: sort jobs by profit in decreasing order
    jobs = sorted(jobs, key=lambda j: j[2], reverse=True)

    max_deadline = max(job[1] for job in jobs)
    slots = [None] * max_deadline    # slots[i] holds job scheduled at time i+1
    total_profit = 0

    for job_id, deadline, profit in jobs:
        # Try to place the job in the latest free slot <= its deadline
        for slot in range(min(deadline, max_deadline) - 1, -1, -1):
            if slots[slot] is None:
                slots[slot] = job_id
                total_profit += profit
                break

    scheduled_jobs = [job for job in slots if job is not None]
    return scheduled_jobs, total_profit


# Driver / Test
if __name__ == "__main__":
    jobs = [("J1", 4, 20), ("J2", 1, 10), ("J3", 1, 40),
             ("J4", 1, 30), ("J5", 3, 50)]

    scheduled_jobs, total_profit = job_sequencing(jobs)
    print(f"Job sequence (in time-slot order): {scheduled_jobs}")
    print(f"Total profit earned: {total_profit}")


Job sequence (in time-slot order): ['J3', 'J5', 'J1']
Total profit earned: 110


### Analysis of the Algorithm
- **Time Complexity:** $O(n^2)$ in the simple array-based implementation shown above ($O(n \log n)$ for sorting plus $O(n \times d)$ for slot search, where $d$ is the maximum deadline); this can be improved to $O(n \log n)$ using a Disjoint Set Union (DSU) to find the latest free slot in near-constant time.
- **Space Complexity:** $O(d)$ where $d$ is the maximum deadline, for the slots array.
- **Remark:** The greedy strategy of considering the most profitable job first, and slotting it as late as possible, guarantees an optimal solution because it never blocks an earlier deadline slot that a lower-profit job might need.


---
## Q3. Program to Implement Prim's Algorithm

### Related Theory
**Prim's Algorithm** finds a **Minimum Spanning Tree (MST)** of a weighted, connected, undirected graph. It grows the MST one vertex at a time, starting from an arbitrary vertex. At each step, it greedily adds the **minimum-weight edge** that connects a vertex already in the MST to a vertex not yet in the MST.

### Related Diagram (Flow)
```
        Start
          |
   Read graph G(V, E) with weights
          |
   Pick an arbitrary start vertex; MST_set = {start}
          |
   +----------------------------------------------------------+
   | while MST_set != V:                                      | <---+
   |   find minimum-weight edge (u, v) with u in MST_set,     |     |
   |   v not in MST_set                                       |     |
   |   add v to MST_set; add edge (u, v) to MST               | ----+
   +----------------------------------------------------------+
          |
   Output MST edges and total weight
          |
         End
```


In [3]:
import heapq

def prims_mst(graph, start):
    """
    graph: dict {vertex: [(neighbor, weight), ...]}
    start: starting vertex
    Returns: (mst_edges, total_weight)
    """
    visited = {start}
    mst_edges = []
    total_weight = 0

    # Min-heap of (weight, u, v) edges available from the visited set
    edge_heap = [(w, start, v) for v, w in graph[start]]
    heapq.heapify(edge_heap)

    while edge_heap and len(visited) < len(graph):
        weight, u, v = heapq.heappop(edge_heap)
        if v in visited:
            continue                      # skip edges that form a cycle
        visited.add(v)
        mst_edges.append((u, v, weight))
        total_weight += weight

        for nxt, w in graph[v]:
            if nxt not in visited:
                heapq.heappush(edge_heap, (w, v, nxt))

    return mst_edges, total_weight


# Driver / Test
if __name__ == "__main__":
    graph = {
        'A': [('B', 2), ('D', 6)],
        'B': [('A', 2), ('C', 3), ('D', 8), ('E', 5)],
        'C': [('B', 3), ('E', 7)],
        'D': [('A', 6), ('B', 8), ('E', 9)],
        'E': [('B', 5), ('C', 7), ('D', 9)],
    }

    mst_edges, total_weight = prims_mst(graph, 'A')
    print("Edges in the Minimum Spanning Tree (Prim's):")
    for u, v, w in mst_edges:
        print(f"  {u} -- {v} : {w}")
    print(f"Total weight of MST: {total_weight}")


Edges in the Minimum Spanning Tree (Prim's):
  A -- B : 2
  B -- C : 3
  B -- E : 5
  A -- D : 6
Total weight of MST: 16


### Analysis of the Algorithm
- **Time Complexity:** $O(E \log V)$ using a binary heap (as implemented above), where $E$ is the number of edges and $V$ the number of vertices; a naive array-based implementation gives $O(V^2)$.
- **Space Complexity:** $O(V + E)$ for the adjacency list and the heap.
- **Remark:** Prim's algorithm is greedy on **vertices** — it grows a single tree outward from a starting vertex — making it well suited to **dense graphs** when implemented with an adjacency matrix.


---
## Q4. Program to Implement Kruskal's Algorithm

### Related Theory
**Kruskal's Algorithm** also finds a Minimum Spanning Tree, but it is greedy on **edges** rather than vertices. All edges are sorted in increasing order of weight, and edges are added to the MST one at a time — as long as adding an edge does **not** form a cycle with the edges already chosen. Cycle detection is efficiently done using a **Disjoint Set Union (Union-Find)** data structure.

### Related Diagram (Flow)
```
        Start
          |
   Read graph G(V, E) with weights
          |
   Sort all edges in increasing order of weight
          |
   Initialize Union-Find: each vertex its own set
          |
   +----------------------------------------------------+
   | for each edge (u, v, w) in sorted order:           | <---+
   |   if find(u) != find(v):                           |     |
   |       add edge to MST; union(u, v)                 |     |
   |   (else: skip -- would form a cycle)               | ----+
   +----------------------------------------------------+
          |
   Output MST edges and total weight
          |
         End
```


In [4]:
class DisjointSet:
    """Union-Find data structure with path compression and union by rank."""
    def __init__(self, vertices):
        self.parent = {v: v for v in vertices}
        self.rank = {v: 0 for v in vertices}

    def find(self, v):
        if self.parent[v] != v:
            self.parent[v] = self.find(self.parent[v])   # path compression
        return self.parent[v]

    def union(self, u, v):
        root_u, root_v = self.find(u), self.find(v)
        if root_u == root_v:
            return False                # already in the same set (would form a cycle)
        if self.rank[root_u] < self.rank[root_v]:
            root_u, root_v = root_v, root_u
        self.parent[root_v] = root_u
        if self.rank[root_u] == self.rank[root_v]:
            self.rank[root_u] += 1
        return True


def kruskals_mst(vertices, edges):
    """
    vertices: list of vertex labels
    edges: list of tuples (u, v, weight)
    Returns: (mst_edges, total_weight)
    """
    edges = sorted(edges, key=lambda e: e[2])   # greedy: sort by increasing weight
    ds = DisjointSet(vertices)
    mst_edges = []
    total_weight = 0

    for u, v, w in edges:
        if ds.union(u, v):               # only add if it doesn't form a cycle
            mst_edges.append((u, v, w))
            total_weight += w

    return mst_edges, total_weight


# Driver / Test
if __name__ == "__main__":
    vertices = ['A', 'B', 'C', 'D', 'E']
    edges = [
        ('A', 'B', 2), ('A', 'D', 6), ('B', 'C', 3),
        ('B', 'D', 8), ('B', 'E', 5), ('C', 'E', 7),
        ('D', 'E', 9),
    ]

    mst_edges, total_weight = kruskals_mst(vertices, edges)
    print("Edges in the Minimum Spanning Tree (Kruskal's):")
    for u, v, w in mst_edges:
        print(f"  {u} -- {v} : {w}")
    print(f"Total weight of MST: {total_weight}")


Edges in the Minimum Spanning Tree (Kruskal's):
  A -- B : 2
  B -- C : 3
  B -- E : 5
  A -- D : 6
Total weight of MST: 16


### Analysis of the Algorithm
- **Time Complexity:** $O(E \log E)$ (equivalently $O(E \log V)$, since $E \le V^2$) — dominated by sorting the edges; Union-Find operations are almost $O(1)$ amortized (inverse Ackermann function) with path compression and union by rank.
- **Space Complexity:** $O(V + E)$ for the edge list and Union-Find structure.
- **Remark:** Kruskal's algorithm is well suited to **sparse graphs**, since its cost is dominated by the number of edges rather than vertices. Both Prim's and Kruskal's produce a valid MST, but may differ in tie-breaking when multiple MSTs exist.


---
## Q5. Program to Implement Dijkstra's Algorithm

### Related Theory
**Dijkstra's Algorithm** solves the **Single-Source Shortest Path** problem for a weighted graph with **non-negative** edge weights. Starting from a source vertex, it greedily selects the unvisited vertex with the smallest known tentative distance, finalizes that distance, and **relaxes** the distances of its neighbors.

### Related Diagram (Flow)
```
        Start
          |
   Read graph G(V, E) with non-negative weights, source vertex
          |
   dist[source] = 0; dist[all others] = infinity
          |
   +--------------------------------------------------------------+
   | while priority queue is not empty:                           | <---+
   |   u = vertex with smallest tentative distance                |     |
   |   for each neighbor v of u with weight w:                    |     |
   |       if dist[u] + w < dist[v]:                              |     |
   |           dist[v] = dist[u] + w  (relax edge)                |     |
   |           push v to priority queue                           | ----+
   +--------------------------------------------------------------+
          |
   Output shortest distance to every vertex from source
          |
         End
```


In [5]:
import heapq

def dijkstra(graph, source):
    """
    graph: dict {vertex: [(neighbor, weight), ...]}
    source: starting vertex
    Returns: dict of shortest distances from source to every vertex
    """
    dist = {v: float('inf') for v in graph}
    dist[source] = 0
    visited = set()
    pq = [(0, source)]        # min-heap of (distance, vertex)

    while pq:
        d, u = heapq.heappop(pq)
        if u in visited:
            continue
        visited.add(u)

        for v, w in graph[u]:
            if v not in visited and dist[u] + w < dist[v]:
                dist[v] = dist[u] + w      # relax the edge
                heapq.heappush(pq, (dist[v], v))

    return dist


# Driver / Test
if __name__ == "__main__":
    graph = {
        'A': [('B', 4), ('C', 1)],
        'B': [('A', 4), ('C', 2), ('D', 5)],
        'C': [('A', 1), ('B', 2), ('D', 8)],
        'D': [('B', 5), ('C', 8)],
    }

    distances = dijkstra(graph, 'A')
    print("Shortest distances from source 'A':")
    for vertex, d in distances.items():
        print(f"  A -> {vertex} : {d}")


Shortest distances from source 'A':
  A -> A : 0
  A -> B : 3
  A -> C : 1
  A -> D : 8


### Analysis of the Algorithm
- **Time Complexity:** $O((V + E) \log V)$ using a binary heap (as implemented above); $O(V^2)$ with a naive array-based implementation, which can be faster for very dense graphs.
- **Space Complexity:** $O(V + E)$ for the adjacency list, distance array, and priority queue.
- **Remark:** Dijkstra's greedy choice (always expanding the closest known vertex) is provably correct **only** when all edge weights are **non-negative**; negative weights require the Bellman-Ford algorithm instead.


---
## Discussion and Conclusion

In this lab, we implemented and analyzed five classical greedy algorithms:

- **Fractional Knapsack** showed a textbook case where the greedy strategy (sorting by value/weight ratio) is always **provably optimal**, because items are divisible.
- **Job Sequencing with Deadlines** demonstrated a greedy strategy over profit ordering combined with a latest-available-slot heuristic, achieving maximum total profit under deadline constraints.
- **Prim's Algorithm** built a Minimum Spanning Tree by greedily growing a single tree vertex-by-vertex, efficient for dense graphs.
- **Kruskal's Algorithm** built a Minimum Spanning Tree by greedily selecting edges in increasing order of weight (using Union-Find for cycle detection), efficient for sparse graphs.
- **Dijkstra's Algorithm** greedily expanded the closest unvisited vertex to compute single-source shortest paths in graphs with non-negative weights.

**Overall Conclusion:**

This lab illustrated that the greedy paradigm, despite its simplicity (always making the locally best choice), produces globally optimal solutions for a wide class of problems that satisfy the **greedy-choice property** and **optimal substructure**. However, greedy algorithms are not universally applicable — problems such as the 0/1 Knapsack Problem require Dynamic Programming instead, since a greedy choice there does not guarantee optimality. Understanding *when* to apply a greedy strategy, and being able to prove its correctness, is a key skill reinforced through this lab.
